# NóiLái — Kaggle TPU v5e-8 runs (vLLM-TPU)

The TPU variant of the T4 notebook, same structure: the three large models (`tpu_main`:
Gemma 3 12B, Qwen3.8-27B, Qwen-SEA-LION-v4.5-27B-IT in bf16), the bf16 drift check
(`bf16_drift_200`: 200 core items per T4 model, unquantized) and the 27B reasoning
sub-study (`reasoning_500_tpu`). Sessions last 9 h (~20 TPU-h/week), one at a time; the
vLLM-TPU startup takes 6–22 minutes per model, so keep `MODELS` short per session and rely
on `--resume`. Attach: accelerator **TPU VM v5e-8**, internet on, the same secrets and
private datasets as the T4 notebook.

Every TPU-hosted model must load under vLLM-TPU and return prompt log-probabilities before its
full run (plan §3): the smoke cell checks that.

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/OWNER/REPO.git"     # [UNCERTAIN: verify]
REPO_REF = "main"
REPO_SUBDIR = "noilai"
BUNDLE_PATH = "/kaggle/input/noilai-bundle/noilai-main.bundle"
CLONE_DIR = "/kaggle/working/repo"
WORK_DIR = "/kaggle/working"
HF_HOME = "/kaggle/tmp/hf"                          # 27B bf16 weights are ~54 GB: not in the 20 GB working dir [UNCERTAIN: verify]
ITEMS_DATASET_DIR = "/kaggle/input/noilai-release"
VERIFY_ITEM_KEYS = ["noilai_test"]

VLLM_TPU_PACKAGE = "vllm-tpu"    # [UNCERTAIN: verify] the TPU build's package name for the pinned version (kaggle-tpu-lab documents the working recipe)
VLLM_TPU_VERSION = None          # [UNCERTAIN: verify] pin once the smoke cell passes; None = latest (recorded by (c))

GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"

RUN_IDS = ["tpu_main"]           # any of: tpu_main, bf16_drift_200, reasoning_500_tpu
MODELS = ["gemma-3-12b-it"]      # subset of each run's models; [] = all
SMOKE_MODEL = "gemma-3-12b-it"
EXTRA_ARGS = ""
DRY_RUN = False

PLATFORM, GPU_TYPE, N_GPUS = "kaggle", "tpu-v5e-8", 1
REQUIRE_TPU = True
RUN_LABEL = "+".join(RUN_IDS)
PUSH_DATASET = True
KAGGLE_DATASET_SLUG = "noilai-runs"
CREATE_DATASET = False

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# (a) environment check: TPU v5e-8 through JAX / tpu-info; the vLLM-TPU startup takes 6-22 minutes on top
import json, shutil, subprocess, sys
print(sys.version.split()[0], "on", sys.platform)
ENV_CHECK = {"tpu_devices": 0, "device_kind": None}
try:
    import jax
    devs = jax.devices()
    ENV_CHECK["tpu_devices"] = len(devs)
    ENV_CHECK["device_kind"] = getattr(devs[0], "device_kind", str(devs[0])) if devs else None
    print("jax", jax.__version__, "devices:", [str(d) for d in devs])
except Exception as e:  # jax missing or no TPU runtime
    print("jax/TPU not visible from here:", repr(e))
if shutil.which("tpu-info"):
    print(subprocess.run(["tpu-info"], capture_output=True, text=True).stdout)
if REQUIRE_TPU and ENV_CHECK["tpu_devices"] == 0:
    raise SystemExit("no TPU device: choose 'TPU VM v5e-8' in the Kaggle accelerator menu and restart")
du = shutil.disk_usage(WORK_DIR)
print(f"disk at {WORK_DIR}: {du.free / 2**30:.1f} GB free of {du.total / 2**30:.1f} GB")
print(json.dumps(ENV_CHECK))

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
        env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
for cand in (REPO_REF, f"origin/{REPO_REF}"):
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF}")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (c) pinned install for the TPU: the vLLM TPU build first, then the project with its eval extra;
# record pip freeze + versions next to the outputs (checklist C4).
import subprocess, sys
from pathlib import Path
spec = f"{VLLM_TPU_PACKAGE}=={VLLM_TPU_VERSION}" if VLLM_TPU_VERSION else VLLM_TPU_PACKAGE
subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]"], check=True)
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
_export(HF_TOKEN_SECRET)
if HF_TOKEN_SECRET != "HF_TOKEN" and os.environ.get(HF_TOKEN_SECRET):
    os.environ["HF_TOKEN"] = os.environ[HF_TOKEN_SECRET]
import colab_setup as CS
ENV_RECORD = CS.record_environment(PROJECT / "data" / "runs" / "env")
print("environment recorded at", ENV_RECORD)

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s): hash against
# configs/run_plan.yaml, canary on every row, row count. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    shutil.copytree(src, PROJECT / "data" / "release", dirs_exist_ok=True)
    print("release copied from", src)
for key in VERIFY_ITEM_KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--key", key], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

### RUN CELL (e) — smoke test on the TPU
`smoke_20` on `SMOKE_MODEL`: the model must load under vLLM-TPU, generate, and return prompt
log-probabilities (the T3 forced-choice scoring needs them).

In [ ]:
# RUN CELL (e): smoke test (smoke_20) on the TPU model.
import shlex
import kaggle_run_plan as KRP
SMOKE_RESULTS = KRP.execute("smoke_20", models=[SMOKE_MODEL], platform=PLATFORM, dry_run=DRY_RUN,
                            continue_on_error=False, extra=shlex.split(EXTRA_ARGS))
for r in SMOKE_RESULTS:
    print(r["model"], "->", r["status"])
if any(str(r["status"]).startswith("failed") for r in SMOKE_RESULTS):
    raise SystemExit("smoke test failed: the TPU route for this model is not ready (fallback: 2xT4 4-bit, labelled quantized)")

### RUN CELL (f) — the TPU runs
One `run_eval.py --resume` command per (run id, model); wall time per model is logged as
TPU device-hours.

In [ ]:
# RUN CELL (f): each run id in RUN_IDS over MODELS (or all of the run's models), --resume.
import json, shlex
import kaggle_run_plan as KRP
RESULTS = []
for run_id in RUN_IDS:
    RESULTS += KRP.execute(run_id, models=MODELS or None, platform=PLATFORM, dry_run=DRY_RUN,
                           continue_on_error=True, extra=shlex.split(EXTRA_ARGS))
print(json.dumps([{"run": r["run"], "model": r["model"], "status": r["status"], "hours": r.get("hours")} for r in RESULTS], indent=1))

In [ ]:
# (g) keep the outputs: copy data/runs (+ compute log) into /kaggle/working (saved with the notebook) and
# add a version to a private Kaggle dataset through scripts/kaggle_dataset.py (credentials from Secrets).
import os, shutil, subprocess, sys
from pathlib import Path
out = Path(WORK_DIR) / "noilai_runs_out"
shutil.copytree(PROJECT / "data" / "runs", out / "runs", dirs_exist_ok=True)
if (PROJECT / "data" / "compute_log.csv").exists():
    shutil.copy2(PROJECT / "data" / "compute_log.csv", out / "compute_log.csv")
print("outputs copied to", out)
if PUSH_DATASET:
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        _export(name)
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        cmd = [sys.executable, "scripts/kaggle_dataset.py", "--src", str(PROJECT / "data" / "runs"),
               "--src", str(PROJECT / "data" / "compute_log.csv"), "--dest", str(Path(WORK_DIR) / "noilai_runs_dataset"),
               "--slug", KAGGLE_DATASET_SLUG, "--message", f"{RUN_LABEL} @ {HEAD[:8]}"]
        if CREATE_DATASET:
            cmd.append("--create")
        subprocess.run(cmd, check=False)
    else:
        print("Kaggle credentials not found: dataset push skipped (outputs are still under", out, ")")

In [ ]:
# (h) GPU-hours log (checklist C1). kaggle_run_plan wrote one row per executed model run; this adds the
# session's remaining wall time (startup, install, model downloads) as "overhead" and prints the totals.
import shutil
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
run_hours = sum((r.get("hours") or 0.0) for r in (globals().get("RESULTS", []) + globals().get("SMOKE_RESULTS", [])))
overhead = max(0.0, CL.hours_since(SESSION_T0) - run_hours)
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=overhead, run_id=f"session-{RUN_LABEL}", purpose="overhead"))
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
out = Path(WORK_DIR) / "noilai_runs_out"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOG, out / "compute_log.csv")
print("log copied to", out / "compute_log.csv", "-> merge into the repository's data/compute_log.csv")